# AIS SFP Distance Prediction Training

Train an image-based plug-tip offset predictor from the locally collected SFP dataset.
Data is expected under `ws_aic/data/distance_prediction/SFP`, and checkpoints are saved under `ws_aic/model/ais_distance_prediction`.


In [1]:
from pathlib import Path
import sys

PROJECT_DIR = Path('/home/whyz/aic_sejong/ws_aic/src/ais/ais_distance_prediction')
WS_ROOT = PROJECT_DIR.parents[2]
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

PROJECT_DIR, WS_ROOT


(PosixPath('/home/whyz/aic_sejong/ws_aic/src/ais/ais_distance_prediction'),
 PosixPath('/home/whyz/aic_sejong/ws_aic'))

In [2]:
from collections import Counter
import math

import torch
from torch.utils.data import DataLoader
from torchvision import transforms

from model import (
    DEFAULT_WEIGHT_ROOT,
    VisionOffsetDataset,
    build_resnet_position_model,
    compute_target_stats,
    distance_sample_has_target,
    distance_target_from_sample,
    evaluate,
    fit,
    load_rpy_randomization_samples,
    rpy_from_sample,
    sample_has_rpy,
    seed_everything,
    split_samples_by_episode,
)

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

/home/whyz/aic_sejong/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device(type='cuda')

In [3]:
DATASET_ROOT = WS_ROOT / 'data' / 'ais_rpy_randomization'
WEIGHT_ROOT = DEFAULT_WEIGHT_ROOT

VERSIONS = ['v3.1', 'v2.3']
CAMERAS = ('left', 'center', 'right')
TARGET_SOURCE = 'actual'
TARGET_MODE = 'plug_reference_in_port'
TARGET_KEYS = ('x_mm', 'y_mm', 'z_mm')
USE_RPY_INPUT = True
RPY_UNIT = 'rad'
RPY_DIM = 3 if USE_RPY_INPUT else 0

IMAGE_SIZE = (256, 288)
BATCH_SIZE = 16
NUM_WORKERS = 8
EPOCHS = 50
LR = 1e-4
WEIGHT_DECAY = 1e-4
BACKBONE = 'resnet50'
PRETRAINED = True
AGGREGATION = 'concat'
NUM_PORT_HEADS = 2
VAL_RATIO = 0.2
EXPAND_ALL_PORTS = False

VERSION_TAG = '_'.join(VERSIONS)
TARGET_TAG = f'{TARGET_SOURCE}_{TARGET_MODE}'
RPY_TAG = '_rpy' if USE_RPY_INPUT else ''
RUN_NAME = f"sfp_distance_{TARGET_TAG}_{BACKBONE}_{'_'.join(CAMERAS)}_{AGGREGATION}_{VERSION_TAG}{RPY_TAG}"

DATASET_ROOT, WEIGHT_ROOT / RUN_NAME

(PosixPath('/home/whyz/aic_sejong/ws_aic/data/ais_rpy_randomization'),
 PosixPath('/home/whyz/aic_sejong/ws_aic/model/ais_distance_prediction/sfp_distance_actual_plug_reference_in_port_resnet50_left_center_right_concat_v3.1_v2.3_rpy'))

In [4]:
samples = load_rpy_randomization_samples(DATASET_ROOT, versions=VERSIONS, splits=('train', 'val'))
if not samples:
    raise FileNotFoundError(f'No samples found under {DATASET_ROOT} for versions={VERSIONS}')

missing_files = []
missing_camera_counts = Counter()
for sample in samples:
    for camera in CAMERAS:
        image_relpath = sample.get('images', {}).get(camera)
        if image_relpath is None:
            missing_camera_counts[camera] += 1
            continue
        image_path = Path(sample['_dataset_root']) / image_relpath
        if not image_path.is_file():
            missing_files.append(image_path)
if missing_files:
    preview = '\n'.join(str(path) for path in missing_files[:5])
    raise FileNotFoundError(f'{len(missing_files)} image files are missing. First missing files:\n{preview}')

complete_samples = [
    sample
    for sample in samples
    if all(camera in sample.get('images', {}) for camera in CAMERAS)
    and distance_sample_has_target(
        sample,
        target_source=TARGET_SOURCE,
        target_mode=TARGET_MODE,
        target_keys=TARGET_KEYS,
    )
    and (not USE_RPY_INPUT or sample_has_rpy(sample))
]
if not complete_samples:
    raise RuntimeError(f'No complete {CAMERAS} distance samples found under {DATASET_ROOT} for versions={VERSIONS}')


def sample_version(sample):
    return sample.get('version') or sample.get('_version', '')


complete_by_version = Counter(sample_version(sample) for sample in complete_samples)
missing_versions = [version for version in VERSIONS if complete_by_version.get(version, 0) == 0]
if missing_versions:
    raise RuntimeError(f'No complete {CAMERAS} samples for versions: {missing_versions}')

train_samples, val_samples, _ = split_samples_by_episode(
    complete_samples,
    val_ratio=VAL_RATIO,
    test_ratio=0.0,
    seed=42,
)
if not train_samples or not val_samples:
    raise RuntimeError(f'Invalid split: train={len(train_samples)} val={len(val_samples)}')

val_samples_by_version = {
    version: [sample for sample in val_samples if sample_version(sample) == version]
    for version in VERSIONS
}
extra_val_sample_sets = {
    version.replace('.', '_'): rows
    for version, rows in val_samples_by_version.items()
    if rows
}

target_stats = compute_target_stats(
    train_samples,
    TARGET_KEYS,
    expand_all_ports=EXPAND_ALL_PORTS,
    target_source=TARGET_SOURCE,
    target_mode=TARGET_MODE,
)
rpy_stats = None
if USE_RPY_INPUT:
    rpy_values = torch.stack([rpy_from_sample(sample, unit=RPY_UNIT) for sample in train_samples])
    rpy_stats = {
        'mean': rpy_values.mean(dim=0),
        'std': rpy_values.std(dim=0, unbiased=False).clamp_min(1e-6),
    }


def target_stack(rows):
    return torch.stack([
        distance_target_from_sample(
            sample,
            target_source=TARGET_SOURCE,
            target_mode=TARGET_MODE,
            target_keys=TARGET_KEYS,
        )
        for sample in rows
    ])


def distance_baselines(rows):
    targets = target_stack(rows)
    zero = torch.linalg.vector_norm(targets, dim=1).mean()
    mean = torch.linalg.vector_norm(targets - target_stats['mean'], dim=1).mean()
    return zero, mean

zero_baseline_distance, mean_baseline_distance = distance_baselines(val_samples)
version_baselines = {
    version: distance_baselines(rows)
    for version, rows in val_samples_by_version.items()
    if rows
}

print(f'dataset_root: {DATASET_ROOT}')
print(f'versions: {VERSIONS}')
print(f'target_source: {TARGET_SOURCE} target_mode: {TARGET_MODE}')
print(f'samples: total={len(samples)} complete_{len(CAMERAS)}view={len(complete_samples)} train={len(train_samples)} val={len(val_samples)}')
print(f'complete_by_version: {dict(complete_by_version)}')
print(f'train_by_version: {dict(Counter(sample_version(sample) for sample in train_samples))}')
print(f'val_by_version: {dict(Counter(sample_version(sample) for sample in val_samples))}')
print(f'missing_camera_counts: {dict(missing_camera_counts)}')
print(f'ports: {Counter(sample.get("port_name", "") for sample in samples)}')
print(f'target_mean_mm: {target_stats["mean"].tolist()}')
print(f'target_std_mm:  {target_stats["std"].tolist()}')
print(f'zero_baseline_distance_mm: {zero_baseline_distance.item():.4f}')
print(f'mean_baseline_distance_mm: {mean_baseline_distance.item():.4f}')
for version, (zero, mean) in version_baselines.items():
    print(f'{version}_zero_baseline_distance_mm: {zero.item():.4f}')
    print(f'{version}_mean_baseline_distance_mm: {mean.item():.4f}')
if USE_RPY_INPUT:
    print(f'rpy_mean_rad: {rpy_stats["mean"].tolist()}')
    print(f'rpy_std_rad:  {rpy_stats["std"].tolist()}')
    print(f'rpy_mean_deg: {(rpy_stats["mean"] * 180.0 / math.pi).tolist()}')
    print(f'rpy_std_deg:  {(rpy_stats["std"] * 180.0 / math.pi).tolist()}')

dataset_root: /home/whyz/aic_sejong/ws_aic/data/ais_rpy_randomization
versions: ['v3.1', 'v2.3']
target_source: actual target_mode: plug_reference_in_port
samples: total=2318 complete_3view=2175 train=1752 val=423
complete_by_version: {'v3.1': 1040, 'v2.3': 1135}
train_by_version: {'v3.1': 795, 'v2.3': 957}
val_by_version: {'v3.1': 245, 'v2.3': 178}
missing_camera_counts: {'right': 120, 'center': 71, 'left': 13}
ports: Counter({'sfp_port_0': 1234, 'sfp_port_1': 1084})
target_mean_mm: [0.9523840546607971, 4.245585918426514, -24.715871810913086]
target_std_mm:  [20.3817081451416, 22.036296844482422, 33.04022216796875]
zero_baseline_distance_mm: 42.6466
mean_baseline_distance_mm: 41.4067
v3.1_zero_baseline_distance_mm: 64.1090
v3.1_mean_baseline_distance_mm: 49.9466
v2.3_zero_baseline_distance_mm: 13.1057
v2.3_mean_baseline_distance_mm: 29.6523
rpy_mean_rad: [-6.945846507733222e-06, 0.0003025673213414848, -0.0032221584115177393]
rpy_std_rad:  [0.08905679732561111, 0.09376295655965805, 0.1

In [5]:
train_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

eval_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

In [6]:
train_dataset = VisionOffsetDataset(
    DATASET_ROOT,
    samples=train_samples,
    cameras=CAMERAS,
    target_keys=TARGET_KEYS,
    target_source=TARGET_SOURCE,
    target_mode=TARGET_MODE,
    transform=train_transform,
    target_mean=target_stats['mean'],
    target_std=target_stats['std'],
    expand_all_ports=EXPAND_ALL_PORTS,
    use_rpy=USE_RPY_INPUT,
    rpy_unit=RPY_UNIT,
    rpy_mean=None if rpy_stats is None else rpy_stats['mean'],
    rpy_std=None if rpy_stats is None else rpy_stats['std'],
)
val_dataset = VisionOffsetDataset(
    DATASET_ROOT,
    samples=val_samples,
    cameras=CAMERAS,
    target_keys=TARGET_KEYS,
    target_source=TARGET_SOURCE,
    target_mode=TARGET_MODE,
    transform=eval_transform,
    target_mean=target_stats['mean'],
    target_std=target_stats['std'],
    expand_all_ports=EXPAND_ALL_PORTS,
    use_rpy=USE_RPY_INPUT,
    rpy_unit=RPY_UNIT,
    rpy_mean=None if rpy_stats is None else rpy_stats['mean'],
    rpy_std=None if rpy_stats is None else rpy_stats['std'],
)
extra_val_datasets = {
    name: VisionOffsetDataset(
        DATASET_ROOT,
        samples=rows,
        cameras=CAMERAS,
        target_keys=TARGET_KEYS,
        target_source=TARGET_SOURCE,
        target_mode=TARGET_MODE,
        transform=eval_transform,
        target_mean=target_stats['mean'],
        target_std=target_stats['std'],
        expand_all_ports=EXPAND_ALL_PORTS,
        use_rpy=USE_RPY_INPUT,
        rpy_unit=RPY_UNIT,
        rpy_mean=None if rpy_stats is None else rpy_stats['mean'],
        rpy_std=None if rpy_stats is None else rpy_stats['std'],
    )
    for name, rows in extra_val_sample_sets.items()
}
if len(train_dataset) == 0 or len(val_dataset) == 0:
    raise RuntimeError(f'No complete samples after camera filtering: train={len(train_dataset)} val={len(val_dataset)}')
print(f'complete {CAMERAS} samples: train={len(train_dataset)} val={len(val_dataset)}')
print('extra val:', {name: len(dataset) for name, dataset in extra_val_datasets.items()})

pin_memory = device.type == 'cuda'
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
    drop_last=False,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
    drop_last=False,
)
extra_val_loaders = {
    name: DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=pin_memory,
        drop_last=False,
    )
    for name, dataset in extra_val_datasets.items()
}

batch = next(iter(train_loader))
batch['image'].shape, batch['target'].shape, batch['port_id'][:8], batch['raw_target'][:3], batch.get('rpy', torch.empty(0))[:3]

complete ('left', 'center', 'right') samples: train=1752 val=423
extra val: {'v3_1': 245, 'v2_3': 178}


(torch.Size([16, 3, 3, 256, 288]),
 torch.Size([16, 3]),
 tensor([0, 0, 1, 0, 0, 0, 0, 0]),
 tensor([[-3.4859e-01,  1.8038e+00,  3.8320e+00],
         [-6.9374e-02,  9.4524e+00,  1.0480e-01],
         [ 1.5378e+01, -6.5362e+00, -8.8461e+01]]),
 tensor([[ 0.0581, -0.2145,  0.1282],
         [ 0.1874, -0.0197, -0.1541],
         [ 0.4309, -1.7006, -1.4479]]))

In [7]:
model = build_resnet_position_model(
    backbone_name=BACKBONE,
    pretrained=PRETRAINED,
    output_dim=len(TARGET_KEYS),
    hidden_dim=256,
    dropout=0.1,
    aggregation=AGGREGATION,
    num_views=len(CAMERAS),
    num_port_heads=NUM_PORT_HEADS,
    rpy_dim=RPY_DIM,
)
model.to(device)

optimizer = torch.optim.AdamW(
    (p for p in model.parameters() if p.requires_grad),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=3,
)

sum(p.numel() for p in model.parameters() if p.requires_grad)


26722374

In [8]:
config = {
    'dataset_root': str(DATASET_ROOT),
    'versions': VERSIONS,
    'cameras': CAMERAS,
    'target_source': TARGET_SOURCE,
    'target_mode': TARGET_MODE,
    'target_keys': TARGET_KEYS,
    'target_mean': target_stats['mean'].tolist(),
    'target_std': target_stats['std'].tolist(),
    'use_rpy_input': USE_RPY_INPUT,
    'rpy_unit': RPY_UNIT,
    'rpy_dim': RPY_DIM,
    'rpy_mean': None if rpy_stats is None else rpy_stats['mean'].tolist(),
    'rpy_std': None if rpy_stats is None else rpy_stats['std'].tolist(),
    'image_size': IMAGE_SIZE,
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'lr': LR,
    'weight_decay': WEIGHT_DECAY,
    'backbone': BACKBONE,
    'pretrained': PRETRAINED,
    'aggregation': AGGREGATION,
    'num_views': len(CAMERAS),
    'num_port_heads': NUM_PORT_HEADS,
    'expand_all_ports': EXPAND_ALL_PORTS,
    'split': {
        'name': 'episode_holdout',
        'val_ratio': VAL_RATIO,
        'val_count': len(val_samples),
        'val_count_by_version': {version: len(rows) for version, rows in val_samples_by_version.items()},
    },
}

history = fit(
    model,
    train_loader,
    val_loader,
    optimizer,
    device,
    epochs=EPOCHS,
    weight_dir=WEIGHT_ROOT,
    run_name=RUN_NAME,
    scheduler=scheduler,
    target_mean=target_stats['mean'],
    target_std=target_stats['std'],
    config=config,
    extra_val_loaders=extra_val_loaders,
)
print(f'best checkpoint: {WEIGHT_ROOT / RUN_NAME / "best.pt"}')
history[-1]

epochs:   2%|▏         | 1/50 [01:21<1:06:29, 81.42s/it, train_loss=0.3038, val_euclidean=28.983, val_loss=0.2893]

epoch=001 train_loss=0.3038 val_loss=0.2893 val_mae=14.343 val_euclidean=28.983 val_v3_1_euclidean=36.889 val_v2_3_euclidean=18.103


epochs:   4%|▍         | 2/50 [01:51<41:08, 51.43s/it, train_loss=0.1816, val_euclidean=24.946, val_loss=0.2256]  

epoch=002 train_loss=0.1816 val_loss=0.2256 val_mae=12.741 val_euclidean=24.946 val_v3_1_euclidean=31.547 val_v2_3_euclidean=15.861


epochs:   6%|▌         | 3/50 [02:22<32:56, 42.06s/it, train_loss=0.1106, val_euclidean=21.556, val_loss=0.1781]

...

epochs:  98%|█████████▊| 49/50 [26:02<00:30, 30.95s/it, train_loss=0.0077, val_euclidean=19.325, val_loss=0.1384]

epoch=049 train_loss=0.0077 val_loss=0.1384 val_mae=9.563 val_euclidean=19.325 val_v3_1_euclidean=19.465 val_v2_3_euclidean=19.133


epochs: 100%|██████████| 50/50 [26:33<00:00, 31.88s/it, train_loss=0.0081, val_euclidean=19.290, val_loss=0.1379]

epoch=050 train_loss=0.0081 val_loss=0.1379 val_mae=9.540 val_euclidean=19.290 val_v3_1_euclidean=19.366 val_v2_3_euclidean=19.186
best checkpoint: /home/whyz/aic_sejong/ws_aic/model/ais_distance_prediction/sfp_distance_actual_plug_reference_in_port_resnet50_left_center_right_concat_v3.1_v2.3_rpy/best.pt


{'epoch': 50.0,
 'train_loss': 0.00805847427050783,
 'val_loss': 0.13787505018742793,
 'val_mae': 9.540318489074707,
 'val_rmse': 13.377911567687988,
 'val_euclidean': 19.289981842041016,
 'val_v3_1_loss': 0.1507721998861858,
 'val_v3_1_mae': 9.72524642944336,
 'val_v3_1_rmse': 12.899950981140137,
 'val_v3_1_euclidean': 19.365821838378906,
 'val_v2_3_loss': 0.12012434344780579,
 'val_v2_3_mae': 9.285745620727539,
 'val_v2_3_rmse': 14.009123802185059,
 'val_v2_3_euclidean': 19.18555450439453}

In [9]:
def with_mm_metrics(metrics):
    return metrics


val_metrics_by_set = {
    'overall': with_mm_metrics(evaluate(
        model,
        val_loader,
        device,
        target_mean=target_stats['mean'],
        target_std=target_stats['std'],
    )),
}
for name, loader in extra_val_loaders.items():
    val_metrics_by_set[name] = with_mm_metrics(evaluate(
        model,
        loader,
        device,
        target_mean=target_stats['mean'],
        target_std=target_stats['std'],
    ))
val_metrics_by_set

{'overall': {'loss': 0.13787505018742793,
  'mae': 9.540318489074707,
  'rmse': 13.377911567687988,
  'euclidean': 19.289981842041016},
 'v3_1': {'loss': 0.1507721998861858,
  'mae': 9.72524642944336,
  'rmse': 12.899950981140137,
  'euclidean': 19.365821838378906},
 'v2_3': {'loss': 0.12012434344780579,
  'mae': 9.285745620727539,
  'rmse': 14.009123802185059,
  'euclidean': 19.18555450439453}}

In [10]:
model.eval()
batch = next(iter(val_loader))
with torch.inference_mode():
    rpy = batch.get('rpy')
    rpy = None if rpy is None else rpy.to(device)
    pred = model(batch['image'].to(device), batch['port_id'].to(device), rpy).cpu()
pred_mm = pred * target_stats['std'] + target_stats['mean']
preview = torch.cat([pred_mm[:8], batch['raw_target'][:8]], dim=1)
print('columns: pred_x pred_y pred_z target_x target_y target_z')
display(preview)

correct_count = 0
total_count = 0
before_distances = []
after_distances = []

with torch.inference_mode():
    for eval_batch in val_loader:
        rpy = eval_batch.get('rpy')
        rpy = None if rpy is None else rpy.to(device)
        pred = model(eval_batch['image'].to(device), eval_batch['port_id'].to(device), rpy).cpu()
        pred_mm = pred * target_stats['std'] + target_stats['mean']
        target_mm = eval_batch['raw_target']

        before_distance = torch.linalg.vector_norm(target_mm, dim=1)
        after_distance = torch.linalg.vector_norm(target_mm - pred_mm, dim=1)
        improved = after_distance < before_distance

        correct_count += int(improved.sum())
        total_count += int(improved.numel())
        before_distances.append(before_distance)
        after_distances.append(after_distance)

before_distance_mm = torch.cat(before_distances)
after_distance_mm = torch.cat(after_distances)
improvement_metrics = {
    'criterion': 'after_distance_mm < before_distance_mm',
    'accuracy': correct_count / max(total_count, 1),
    'correct': correct_count,
    'total': total_count,
    'before_distance_mean_mm': float(before_distance_mm.mean()),
    'after_distance_mean_mm': float(after_distance_mm.mean()),
    'distance_improvement_mean_mm': float((before_distance_mm - after_distance_mm).mean()),
}
improvement_metrics

columns: pred_x pred_y pred_z target_x target_y target_z


tensor([[  7.8369,  -9.0070, -26.2362,   1.3334,  -7.9671, -29.8365],
        [ 15.5195, -16.7312, -14.6333,  14.6878, -23.6943, -10.1376],
        [ 14.5518,  -8.5777, -22.7857,  15.2661, -33.6925,  -8.6592],
        [  0.9417,  40.9976, -26.0138,  -7.7968,  58.8404, -14.2243],
        [ 10.1114,  45.7686, -27.8371,  18.2936,  66.1049, -21.3454],
        [ -9.4403, -28.5119, -11.1061, -18.4431, -44.1523,  -1.2004],
        [  8.4425, -38.8657, -26.8644,   4.5197, -57.6709, -29.0143],
        [ 12.6605, -19.7080, -30.6428,  10.7402,  -1.7067, -32.6576]])

{'criterion': 'after_distance_mm < before_distance_mm',
 'accuracy': 0.6784869976359338,
 'correct': 287,
 'total': 423,
 'before_distance_mean_mm': 42.64663314819336,
 'after_distance_mean_mm': 19.289981842041016,
 'distance_improvement_mean_mm': 23.356643676757812}